## 1 - Loading Processed Data

In this step, we load processed video dataset from the previously notebooks


In [1]:
import pandas as pd
import os

In [2]:
print("--- LOADING PROCESSED DATA ---")

load_path = "./processed_images/fei_images_final.csv"
fei_imgs = pd.read_csv(load_path)

print(f"{len(fei_imgs)} images loaded!")
pd.set_option('display.max_colwidth', None)
display(fei_imgs.sample(5))

load_path = "./processed_images/celeb_frames.csv"
celeb_imgs = pd.read_csv(load_path)

print(f"{len(celeb_imgs)} images loaded!")
pd.set_option('display.max_colwidth', None)
display(celeb_imgs.sample(5))

--- LOADING PROCESSED DATA ---
8054 images loaded!


,path,label,split,dataset,method,target,source
5451,/home/lucia_pola/internship-deepfake-forensic/deepfake-forensics-pipeline/02_Extended_Framework/Datasets/FEI_MORPHV2_DATASET/train/fake/M_48-11_112-11_C15_B50_W50_PA15_PM00_F00_ssd.png,1,train,FEI,C15,48,112
1967,/home/lucia_pola/internship-deepfake-forensic/deepfake-forensics-pipeline/02_Extended_Framework/Datasets/FEI_MORPHV2_DATASET/train/fake/M_48-11_112-11_C05_B50_W50_PA05_PM00_F00_ssd.png,1,train,FEI,C05,48,112
6003,/home/lucia_pola/internship-deepfake-forensic/deepfake-forensics-pipeline/02_Extended_Framework/Datasets/FEI_MORPHV2_DATASET/train/fake/M_79-11_170-11_C08_B50_W50_PA08_PM00_F00_ssd.png,1,train,FEI,C08,79,170
4956,/home/lucia_pola/internship-deepfake-forensic/deepfake-forensics-pipeline/02_Extended_Framework/Datasets/FEI_MORPHV2_DATASET/train/fake/M_159-11_188-11_C08_B50_W50_PA08_PM00_F00_ssd.png,1,train,FEI,C08,159,188
1194,/home/lucia_pola/internship-deepfake-forensic/deepfake-forensics-pipeline/02_Extended_Framework/Datasets/FEI_MORPHV2_DATASET/train/fake/M_46-11_45-11_C02_B50_W50_PA02_PM00_F00_ssd.png,1,train,FEI,C02,46,45


162255 images loaded!


,path,label,split,dataset,method,target,source
93198,CELEBDFV3_DATASET/train/fake/id06913_id17_IP_LAP_id17_0008_test_id06913_CFoP6f5lk7M_f0_ssd.jpg,1,train,Celeb-DF-v3,IP_LAP,id06913,id17
61969,CELEBDFV3_DATASET/train/fake/id03839_id16_EDTalk_id16_0001_test_id03839_ajkGXKUvTWY_f1_ssd.jpg,1,train,Celeb-DF-v3,EDTalk,id03839,id16
135173,CELEBDFV3_DATASET/train/fake/id29_id28_Celeb-DF-v2_id28_id29_0002_f2_ssd.jpg,1,train,Celeb-DF-v3,Celeb-DF-v2,id29,id28
55563,CELEBDFV3_DATASET/train/fake/id04119_id33_Real3DPortrait_id33_0006_test_id04119_4Qb-pd4uKyQ_f0_ssd.jpg,1,train,Celeb-DF-v3,Real3DPortrait,id04119,id33
131968,CELEBDFV3_DATASET/train/fake/id31_id3_Celeb-DF-v2_id3_id31_0002_f1_ssd.jpg,1,train,Celeb-DF-v3,Celeb-DF-v2,id31,id3


## 2 - Master Dataset Compilation & Data Export

In this final preprocessing step, we consolidate our distinct datasets into standardized structures:

1. **Format Standardization:** We unify the column structure (`path`, `label`, `split`, `dataset`, `method`) across all dataframes and explicitly cast the labels into standard integers (`0` for Real, `1` for Fake).
2. **Master Dataset Assembly (FEI + Celeb-DF-v3):** We concatenate the FEI and Celeb-DF-v3 dataframes into a single, shuffled dataset. Each dataset keeps its own identity-based **Train**, **Validation** and **Test** split (computed in the respective preprocessing notebooks), so both datasets contribute to training, validation and testing. This `master_df` is exported as `master_dataset.csv`.
3. **No External Holdout:** Unlike the earlier Benchmark-phase pipeline (FF++-based), Celeb-DF-v3 is **not** held out here as a separate external test set — it is merged into the same train/val/test structure as FEI to increase training diversity. Cross-dataset generalization is therefore not measured by this master dataset; a dedicated external holdout would need to be reintroduced if that evaluation is required again.
4. **Final Audit:** We output grouped distribution tables and random samples to verify the final dataset composition, split proportions, and label balancing.

In [3]:
# Prepare FEI
df_fei = fei_imgs[['path', 'label', 'split', 'method', 'target', 'source']].copy()
df_fei['dataset'] = 'FEI'
df_fei['label'] = df_fei['label'].replace({'original': 0, 'fake': 1}).astype(int)

# Prepare CELEB
df_celeb = celeb_imgs[['path', 'label', 'split', 'method', 'target', 'source']].copy()
df_celeb['dataset'] = 'Celeb-DF'
df_celeb['label'] = df_celeb['label'].astype(int)

# Merge
master_df = pd.concat([df_fei, df_celeb], ignore_index=True)
master_df = master_df.sample(frac=1, random_state=42).reset_index(drop=True)
master_df.to_csv("master_dataset.csv", index=False)

print("Merge completed! File saved as 'master_dataset.csv'.")

print("\n--- MASTER DATASET DISTRIBUTION (FEI + CELEB) ---")
distribution_master = master_df.groupby(['dataset', 'method', 'split', 'label']).size().unstack(fill_value=0)

if len(distribution_master.columns) == 2:
    distribution_master.columns = ['0 (Real)', '1 (Fake)']
display(distribution_master)

print("Master Dataset Sample:")
display(master_df.sample(5))

Merge completed! File saved as 'master_dataset.csv'.

--- MASTER DATASET DISTRIBUTION (FEI + CELEB) ---


0 (Real)  1 (Fake)
dataset  method    split                    
Celeb-DF AniTalker test          0       255
                   train         0      8301
                   val           0       294
         BlendFace train         0      5388
                   val           0       456
...                            ...       ...
FEI      C16       train         0      1046
                   val           0        30
         original  test         30         0
                   train       140         0
                   val          30         0

[78 rows x 2 columns]

Master Dataset Sample:


,path,label,split,method,target,source,dataset
87204,CELEBDFV3_DATASET/train/fake/id24_id19_InSwapper_id19_id24_0008_f1_ssd.jpg,1,train,InSwapper,id24,id19,Celeb-DF
24418,CELEBDFV3_DATASET/train/fake/id21_id28_TPSMM_id28_id21_0008_f1_ssd.jpg,1,train,TPSMM,id21,id28,Celeb-DF
152654,CELEBDFV3_DATASET/train/fake/id2_id1_GHOST_id1_id2_0005_f2_ssd.jpg,1,train,GHOST,id2,id1,Celeb-DF
167264,CELEBDFV3_DATASET/train/fake/id23_id38_HifiFace_id38_id23_0009_f0_ssd.jpg,1,train,HifiFace,id23,id38,Celeb-DF
122295,/home/lucia_pola/internship-deepfake-forensic/deepfake-forensics-pipeline/02_Extended_Framework/Datasets/FEI_MORPHV2_DATASET/train/fake/M_112-11_81-11_C08_B50_W50_PA08_PM00_F00_ssd.png,1,train,C08,112,81,FEI


## 3 - Saving Data & Exporting Archive

To conclude this notebook, we first save our fully cleaned and processed DataFrames to local CSV files (e.g., `master_dataset.csv`). This ensures our prepared metadata is safely stored and ready to be directly loaded into the next notebook of our pipeline without needing to re-run the intensive preprocessing and extraction steps.

Finally, we package the entire structured dataset into a single, highly portable archive (`deepfake_dataset.zip`). This makes it easy to download, store, or transfer the data for model training.

**Contents of the final archive:**
* `FEI_MORPHV2_DATASET/`: The processed and split FEI dataset frames.
* `CELEBDFV3_DATASET/`: The processed and split Celeb-DF dataset frames.
* `master_dataset.csv`: The unified metadata for the training, validation, and test sets.

*(Note: We use the `-r` flag to include all subdirectories recursively and the `-q` flag to run the compression quietly, keeping the notebook output clean).*

In [4]:
print("--- SAVING DATAFRAME ---")

save_path_master = "master_dataset.csv"

master_df.to_csv(save_path_master, index=False)

print(f"Master Data successfully saved to: {save_path_master}")

--- SAVING DATAFRAME ---
Master Data successfully saved to: master_dataset.csv


In [ ]:
!zip -rq deepfake_dataset.zip  FEI_MORPHV2_DATASET CELEBDFV3_DATASET master_dataset.csv